# TRIAGE-EG Dataset Survey

**This is a bounded layout survey, not a complete Data Audit.**

Notebook này tự tìm repository trong `/kaggle/working`; nếu chưa có, nó clone branch/ref được cấu hình từ GitHub. Survey chỉ đọc `/kaggle/input`, không decode video và chỉ ghi report nhỏ vào `/kaggle/working/dataset_survey`.

In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
DATASET_ROOT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
OUTPUT_ROOT = Path(os.environ.get("AIC_SURVEY_OUTPUT_ROOT", "/kaggle/working/dataset_survey"))
REFRESH_REPO = os.environ.get("AIC_REFRESH_REPO", "0") == "1"

print("repo URL:", REPO_URL)
print("repo ref:", REPO_REF)
print("repo directory:", REPO_DIR)
print("dataset:", DATASET_ROOT)
print("output:", OUTPUT_ROOT)
print("python:", platform.python_version())

## Repository bootstrap

Cell này rerun-safe: không clone đè lên repository đã tồn tại. Đặt `AIC_REPO_REF` thành commit SHA nếu cần tái lập chính xác; đặt `AIC_REFRESH_REPO=1` nếu muốn fetch ref mới vào checkout đã có. Repository private cần Kaggle secret/credential Git đã được cấu hình và không được in token.

In [ ]:
def run_git(*args: str, cwd: Path | None = None) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        ["git", *args], cwd=cwd, check=True, text=True, capture_output=True
    )

if (REPO_DIR / ".git").is_dir():
    print("Using existing repository checkout.")
    if REFRESH_REPO:
        print("Refreshing requested ref from origin...")
        run_git("fetch", "--depth", "1", "origin", REPO_REF, cwd=REPO_DIR)
        run_git("checkout", "--detach", "FETCH_HEAD", cwd=REPO_DIR)
    else:
        print("Keeping the existing detached HEAD; no network refresh requested.")
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"REPO_DIR exists but is not a Git checkout: {REPO_DIR}")
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    print("Cloning repository...")
    run_git("clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR))
    run_git("fetch", "--depth", "1", "origin", REPO_REF, cwd=REPO_DIR)
    run_git("checkout", "--detach", "FETCH_HEAD", cwd=REPO_DIR)

resolved_commit = run_git("rev-parse", "HEAD", cwd=REPO_DIR).stdout.strip()
print("resolved git commit:", resolved_commit)

In [ ]:
module_path = REPO_DIR / "src" / "triage_eg" / "data" / "dataset_survey.py"
if not module_path.is_file():
    raise RuntimeError(
        "dataset_survey.py is missing from the selected Git ref. "
        "Commit/push the dataset-survey files, then set AIC_REPO_REF to that branch or commit. "
        f"Selected ref={REPO_REF!r}, commit={resolved_commit!r}."
    )

src_path = str(REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from triage_eg.data.dataset_survey import (
    SurveyLimits,
    survey_dataset,
    write_survey_outputs,
)

print("Loaded survey module:", module_path)

In [ ]:
limits = SurveyLimits(
    max_depth=4,
    max_listed_per_directory=20,
    max_examples_per_group=5,
    max_csv_rows=20,
    max_json_bytes=1_048_576,
    max_npy_rows=5,
    max_stat_operations=5_000,
)
result = survey_dataset(DATASET_ROOT, limits=limits, strict_root=True, seed=2026)
artifact_paths = write_survey_outputs(result, OUTPUT_ROOT)
print(result.summary["disclaimer"])

## Bounded root tree

In [ ]:
result.summary["root_entries"]

## Asset groups

In [ ]:
result.summary["asset_groups"]

## Schema observations

In [ ]:
result.summary["schema_observations"]

## Cross-asset sample associations

In [ ]:
result.summary["mapping_observations"]["cross_asset_samples"]

## Sample issues and unknowns

In [ ]:
{"issues": result.summary["sample_issues"], "unknowns": result.summary["unknowns"]}

In [ ]:
print("This is a bounded layout survey, not a complete Data Audit.")
for name, path in artifact_paths.items():
    print(f"{name}: {path}")

## Download one survey bundle

Chạy cell dưới cùng sau khi survey hoàn tất. ZIP chỉ chứa ba report nhỏ cần gửi để phân tích bước tiếp theo; không chứa video, ảnh, vector hoặc dữ liệu nguồn.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

required_reports = [
    OUTPUT_ROOT / "dataset_survey.json",
    OUTPUT_ROOT / "dataset_survey.md",
    OUTPUT_ROOT / "sample_inventory.jsonl",
]
missing_reports = [path for path in required_reports if not path.is_file()]
if missing_reports:
    raise RuntimeError(
        "Survey chưa sinh đủ report: " + ", ".join(str(path) for path in missing_reports)
    )

bundle_path = OUTPUT_ROOT.parent / "dataset_survey_bundle.zip"
with ZipFile(bundle_path, mode="w", compression=ZIP_DEFLATED) as archive:
    for report_path in required_reports:
        archive.write(report_path, arcname=report_path.name)

print("Download this single file and send it for the next analysis step:")
print(bundle_path)
print("Included:", [path.name for path in required_reports])